In [1]:
import os
from gliner2 import GLiNER2
#from spacy import displacy
import json
import torch
import re

2026-08-25 13:55:16.145834607 [W:onnxruntime:Default, device_discovery.cc:325 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:92 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
SOURCE_DIR = "pb"
DEST_DIR = "pb_done"

In [ ]:
from openai import OpenAI


def add_missing_periods(text: str, model: str = "nvidia/nemotron-3-super-120b-a12b:free") -> str:
    client = OpenAI(
        api_key="",
        base_url="https://openrouter.ai/api/v1",
        default_headers={
            "HTTP-Referer": "http://localhost",
            "X-Title": "Punctuation Fixer",
        },
    )

    system_prompt = (
        "You are an expert secretary of the League of Nations. The input sentence may be missing periods or final punctuation. Your task is to add the missing periods and correct the final punctuation without changing the meaning. If you think that a end of line should be present, add a period instead"
        "Return only the corrected sentence, with no explanation."
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ],
        temperature=0,
    )

    return response.choices[0].message.content.strip()

In [4]:
txt_list = os.listdir(SOURCE_DIR)
os.makedirs(DEST_DIR, exist_ok = True)

In [5]:
extractor_gpu1 = GLiNER2.from_pretrained("fastino/gliner2-multi-v1").to("cuda")

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

🧠 Model Configuration
Encoder model      : microsoft/mdeberta-v3-base
Counting layer     : count_lstm
Token pooling      : first


In [6]:
schema = extractor_gpu1.create_schema().entities({
        # Temporel
        "date":             "Explicit dates or time references (ex: '10 June 2024', '1948', 'Monday 12th October 1972')",
        "session number":   "Meeting or session identifier (ex: '6826th meeting', '78th session')",

        # Personnes & acteurs
        "person":           "Name of an individual (ex: 'António Guterres', 'Mr. De Rivière')",
        "role":             "Institutional title or function (ex: 'Secretary-General', 'Permanent Representative')",
        "speaker":          "Person actively taking the floor, as opposed to persons merely mentioned",
        "population group": "People, ethnic group or population category (ex: 'Palestinian people', 'Rohingya', 'civilians')",

        # Géopolitique
        "country":          "Sovereign state or territory (ex: 'France', 'Palestine', 'Western Sahara')",
        "city":             "City or precise geographic location (ex: 'Kyiv', 'Geneva', 'Rafah')",
        "region":           "Supranational or subnational area (ex: 'Middle East', 'Sahel')",

        # Institutions
        "organization":     "Any institutional body: UN organs, agencies, NGOs, alliances (ex: 'UNHCR', 'NATO', 'Security Council'), governements",
        "political party":  "Political party or movement (ex: 'Communist Party of Romania', 'Hezbollah')",

        # Documents & actes
        "resolution":           "Resolution by name (ex: 'Resolution 2735 (2024)')",
        "document reference":   "Formal UN document symbol (ex: 'S/2024/123', 'A/78/PV.10')",
        "international treaty": "International agreement or convention (ex: 'Geneva Convention', 'NPT')",
        "legal norm":       "Principle or norm of international law, not a formal treaty (ex: 'right to self-determination', 'responsibility to protect')",
        "budget or funding": "Financial amount linked to a mandate or humanitarian appeal (ex: '$2.3 billion humanitarian appeal')",

        # Procédure & votes
        "vote result":                   "Numerical vote outcome (ex: '14 votes in favour, 1 abstention')",
        "agenda item":                   "Item on the official agenda (ex: 'The situation in the Middle East')",
        "decision":                      "Formal decision or conclusion adopted (ex: 'The Council decided to extend the mandate')",
        "diplomatic position or demand": "Explicit stance or demand by an actor (ex: 'calls for an immediate ceasefire')",
        "sanction regime":  "Identified sanctions regime or committee (ex: '1267 Committee', 'DPRK sanctions regime')",
        

        # Situations & événements
        "situation": "Ongoing geopolitical context or crisis (ex: 'war in Ukraine', 'humanitarian crisis in Sudan')",
        "event":     "Punctual time-bounded occurrence (ex: 'airstrike on Rafah', 'peace talks in Cairo')",
        "coalition":        "Informal bloc or group of states (ex: 'P5', 'G77', 'Western bloc')",
        "humanitarian issue": "Humanitarian or human rights theme (ex: 'food insecurity', 'sexual violence', 'displacement')",
        "operation":        "Military, peacekeeping or field operation (ex: 'MINUSMA', 'UNIFIL', 'Operation Prosperity Guardian')",
    })

In [7]:
def make_chunk_sentence(text, n=10):
    sentences = text.split('.')
    tmp = []
    chunks = []
    counter = 0
    if n <= 0:
        return
    for sentence in sentences:
        if counter < n:
            tmp.append(sentence)
            counter += 1
        else:
            chunk = '. '.join(tmp) +'.'
            chunks.append(chunk)
            tmp = [sentence]
            counter = 1
    
    if tmp:
        chunk = '. '.join(tmp)+'.'
        chunks.append(chunk)
    
    return chunks

In [8]:
def correct_span_index(prediction, n):
    for entity_list in prediction['entities'].values():
        for entity in entity_list:
            entity['start'] += n
            entity['end'] += n+1
    return prediction

In [9]:
def merge_predictions(predictions):
    merged = {'entities': {key: [] for key in predictions[0]['entities']}}
    for prediction in predictions:
        for entity_key, entity_list in prediction['entities'].items():
            merged['entities'][entity_key].extend(entity_list)
    return merged

In [10]:
def to_spacy_json(pred, txt):
    entities = []
    for entity_type, entity_list in pred['entities'].items():
        for entity in entity_list:
            entities.append({
                "start": entity['start'],
                "end": entity['end'],
                "label": entity_type
            })
    
    return {
        "text": txt,
        "ents": entities
    }


In [11]:
def extract_entities(extractor, txt):
    reduce_chunk = True
    n = 50 # Nombre initial de phrases par chunk
    original = ''
    while reduce_chunk:
        # Dans le cas où une phrase du texte fait + de 300 mots
        if n == 0:
            n = 1
            #raise OverflowError('A sentence of the document is higher than 340 words.')
            

        chunks = make_chunk_sentence(txt, n)
        reduce_chunk = False # On suppose que le découpage est valide jusqu'à preuve du contraire

        #Si un des chunks fait plus de 300 mots, on réduit le nombre de phrase par chunk
        for chunk in chunks:
            if len(chunk.split()) > 340:
                reduce_chunk = True
                n -= 1

                if n == 0 and original != chunk:
                    original = chunk
                    print("Correction d'une phrase")
                    try:
                        corrected = add_missing_periods(chunk)
                        txt = txt.replace(chunk, corrected, 1)
                        n = 10
                    except Exception as e:
                        print(e)
                        n = 1
                        reduce_chunk = False
                elif n == 0:
                    n = 1
                    reduce_chunk = False
                break

    recomposed_txt = ''
    results = []
    count = 1
    for chunk in chunks:
        if len(chunk.split()) <= 340:
            result = extractor.extract(
                chunk,
                schema,
                include_spans=True,
                include_confidence=True
            )

            count+=1

            result_corrected = correct_span_index(result, len(recomposed_txt))

            results.append(result_corrected)

        if recomposed_txt:
            recomposed_txt = recomposed_txt + ' ' + chunk
        else:
            recomposed_txt = chunk
        
    return to_spacy_json(merge_predictions(results), recomposed_txt)


In [14]:

def process_files(extractor, files):
    for file in files:
        fbase = '.'.join(file.split('.')[:-1]) + '.json'
        fname = os.path.join(DEST_DIR, fbase)

        print(f"Process file {file}")

        try:
            with open(os.path.join(SOURCE_DIR, file), encoding="UTF-8") as f:
                txt = f.read()
        except Exception as e:
            print(f"Erreur lors de l'ouverture de {file}: {e}")
            continue

        try:
            pred = extract_entities(extractor, txt)
        except Exception as e:
            print(f"Erreur avec {file}: {e}")
            continue

        try:
            with open(fname, "w", encoding="utf-8") as f:
                json.dump(pred, f, ensure_ascii=False, indent=2)
        except Exception as e:
            print(f"Erreur lors de l'enregistrement de {fname}: {e}")


In [15]:
process_files(extractor_gpu1, txt_list)

Process file A_76_PV.56_%28Resumption_1%29-EN.pdf_d23e6be5.txt
Process file A_78_PV.50_%28Resumption_1%29-EN.pdf_3270b7ca.txt
Process file A_75_PV.77_%28Resumption_1%29-EN.pdf_01e842a7.txt
Process file A_RES_52_40%26A_RES_52_40%5EB%5E-EN.pdf_1d47ddc7.txt
Process file A_77_PV.49_%28Resumption_1%29-EN.pdf_d30aa54b.txt
Process file A_S-1_PV%2CVOL-III-EN.pdf_01f7b677.txt
Process file A_77_PV.72_%28Resumption_1%29-EN.pdf_4bcdf801.txt
Process file A_S-1_PV%2CVOL-I-EN.pdf_e5d892ed.txt
Process file A_S-23_PV-10%28Resumption1%29-EN.pdf_32e771d0.txt
Process file A_75_PV.48_%28Resumption_1%29-EN.pdf_886111a0.txt
Process file A_76_PV.54_%28Resumption%29-EN.pdf_157924eb.txt
Process file A_79_PV.55_%28Resumption_1%29-EN.pdf_7cd77021.txt
Process file A_75_PV.3_%28Resumption_1%29-EN.pdf_9818c8cd.txt
Process file A_RES_41_39%5EC%5E--A_RES_41_39-EN.pdf_e36d8a46.txt
Correction d'une phrase
Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}
Process file A_74_PV.32_%28Resumption_1%29-